# STT_MultiLingva — Colab runner

Transcribes a meeting that mixes Russian and English (optionally Armenian) on a
free Colab T4. The model runs on the GPU Colab gives you; nothing is sent to a
transcription API.

## Read this before uploading anything

**Colab is Google infrastructure.** Audio uploaded here leaves your machine and
lands on Google's servers. If the recording is confidential, that is the same
disclosure you were avoiding by not using a transcription API — the model is
local to the runtime, but the file is not.

Use this for audio you are free to share. For a confidential meeting, run the
Docker image on a GPU inside your own perimeter; `README.md` covers it.

**Runtime → Change runtime type → T4 GPU** before running anything.

---

Run sections 1–4, then **section 5** for the drag-and-drop interface. Section 6
is the command-line path, for batch work or scripting.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


## 2. Install

`faster-whisper` pulls CTranslate2, which needs the cuDNN 9 runtime. Colab ships
CUDA but not always the matching cuDNN, so it goes in explicitly.

In [ ]:
!pip install -q faster-whisper==1.2.1 "gradio==6.*"
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import ctypes, glob, os, site

# The nvidia wheels are PEP 420 namespace packages: no __init__.py, so
# nvidia.cudnn.__file__ is None and building a path from it raises TypeError.
# Glob the installed tree instead of importing anything.
roots = set(site.getsitepackages() + [site.getusersitepackages()])
libs = sorted({so for r in roots for so in glob.glob(os.path.join(r, "nvidia", "*", "lib", "*.so*"))})
dirs = sorted({os.path.dirname(so) for so in libs})

if not dirs:
    print("No bundled NVIDIA libraries found. CTranslate2 will try the system ones;")
    print("if loading the model fails, rerun this cell.")
else:
    # Two separate needs, and only doing the first is why this looked fine and
    # then failed. LD_LIBRARY_PATH is read by the dynamic loader when a process
    # starts, so it reaches the `!python transcribe.py` cells below but never
    # this kernel, whose loader fixed its search path long before now.
    os.environ["LD_LIBRARY_PATH"] = ":".join(dirs + [os.environ.get("LD_LIBRARY_PATH", "")])

    # Preloading into the global namespace is what makes the in-notebook
    # interface work: CTranslate2 dlopens cuDNN later and finds it already
    # resolved. Versioned duplicates fail harmlessly, hence the bare count.
    ok = 0
    for so in libs:
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
            ok += 1
        except OSError:
            pass
    print(f"{len(dirs)} NVIDIA lib dir(s), {ok}/{len(libs)} libraries preloaded")


## 3. Write out the tool

The two source files are carried inside this notebook, so nothing is fetched at
run time. That means it works from a private repository, from a Drive copy, or
from a notebook someone emailed you — none of which the previous `git clone`
survived.

They are kept in step by CI: a change to the sources that has not been embedded
here fails the build, so these cells cannot quietly drift from the repository.

In [ ]:
%cd /content


In [ ]:
%%writefile transcribe.py
#!/usr/bin/env python3
"""Transcribe a multilingual meeting recording entirely on local hardware.

Whisper detects the language once, from the first window, and then treats the
whole recording as monolingual. On a meeting where people switch between
Russian and English that locks onto whichever language opened the call and the
other one comes out transliterated, translated, or dropped.

This runs language detection per speech window instead: voice activity
detection splits the audio, each window is classified against an allowed set of
languages, and only then transcribed with that language pinned. Windows the
classifier is unsure about fall back to the primary language rather than
guessing.

Nothing leaves the machine. Model weights are read from a local directory or
the HuggingFace cache; set HF_HUB_OFFLINE=1 once they are in place to make that
guarantee enforceable rather than a promise.
"""

from __future__ import annotations

import argparse
import json
import math
import sys
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
from faster_whisper import WhisperModel
from faster_whisper.audio import decode_audio
from faster_whisper.vad import VadOptions, get_speech_timestamps

SAMPLE_RATE = 16000


@dataclass
class Line:
    start: float
    end: float
    language: str
    language_probability: float
    text: str
    speaker: str | None = None


def group_speech_windows(
    audio: np.ndarray,
    max_window: float,
    min_silence_ms: int,
    speech_pad_ms: int,
    split_silence: float,
) -> list[tuple[int, int]]:
    """Merge VAD speech regions into windows of at most `max_window` seconds.

    Language detection needs a few seconds of speech to be reliable, so isolated
    VAD regions are merged -- but only across pauses short enough to be breath
    within one person's turn. A gap of `split_silence` or more ends the window
    regardless of how much room is left, because that is where a speaker change
    happens, and a window spanning one is decoded entirely in whichever language
    its opening utterance was classified as. Merging on window room alone would
    swallow exactly the Russian-pause-English sequence this tool exists for.
    """
    # Padding is deliberately switched off here and applied at the end. The VAD
    # grows every region by speech_pad_ms on each side, and where the silence
    # between two regions is shorter than twice that, it splits the difference
    # instead -- so the gap this function can see is the real pause minus 0.4s,
    # or zero. Measuring --split-silence against that would silently require a
    # 1.1s pause to mean 0.7, and 0.7-1.1s is ordinary turn-taking: exactly the
    # switch this is supposed to catch.
    regions = get_speech_timestamps(
        audio,
        VadOptions(min_silence_duration_ms=min_silence_ms, speech_pad_ms=0),
        sampling_rate=SAMPLE_RATE,
    )
    if not regions:
        return []

    limit = int(max_window * SAMPLE_RATE)
    split_gap = int(split_silence * SAMPLE_RATE)
    pad = int(speech_pad_ms / 1000 * SAMPLE_RATE)
    windows: list[tuple[int, int]] = []
    start, end = regions[0]["start"], regions[0]["end"]

    for region in regions[1:]:
        gap = region["start"] - end
        if gap < split_gap and region["end"] - start <= limit:
            end = region["end"]
        else:
            windows.extend(split_oversized(start, end, limit))
            start, end = region["start"], region["end"]
    windows.extend(split_oversized(start, end, limit))

    # Now restore the padding the VAD would have added, so a decoded window
    # still carries the leading and trailing moment that keeps Whisper from
    # clipping the first and last word.
    ceiling = len(audio)
    return [(max(0, s - pad), min(ceiling, e + pad)) for s, e in windows]


def split_oversized(start: int, end: int, limit: int) -> list[tuple[int, int]]:
    """Cut a stretch of uninterrupted speech down to the window limit.

    A speaker can hold the floor for minutes without a pause the VAD will call a
    boundary. Such a region has to be cut somewhere arbitrary, because the
    alternative is one enormous window whose language is decided by its opening
    seconds. Pieces are evenly sized rather than limit-then-remainder, which
    would leave a final sliver too short to classify.
    """
    span = end - start
    if span <= limit:
        return [(start, end)]

    pieces = math.ceil(span / limit)
    step = math.ceil(span / pieces)
    return [(cut, min(cut + step, end)) for cut in range(start, end, step)]


def pick_language(
    model: WhisperModel,
    chunk: np.ndarray,
    allowed: list[str],
    primary: str,
    threshold: float,
) -> tuple[str, float]:
    """Classify one window, restricted to the languages we expect to hear.

    Whisper ranks all 99 languages it knows. On a Russian meeting that regularly
    surfaces Ukrainian or Bulgarian as a near-miss, so the ranking is filtered to
    the allowed set before picking a winner. Below `threshold` the window is not
    distinctive enough to trust -- short interjections score badly no matter the
    model -- and the primary language is used instead.
    """
    _, _, all_probs = model.detect_language(audio=chunk)
    scores = {lang: prob for lang, prob in all_probs if lang in allowed}
    if not scores:
        return primary, 0.0

    best = max(scores, key=scores.get)
    if scores[best] < threshold:
        # The primary's own score, not the rejected candidate's. Reporting
        # en=0.55 as "ru, p=0.55" would make the JSON say the classifier was
        # fairly confident about Russian when it was not confident about
        # anything -- and that field is what you tune --threshold against.
        return primary, scores.get(primary, 0.0)
    return best, scores[best]


def transcribe(
    model: WhisperModel,
    audio: np.ndarray,
    windows: list[tuple[int, int]],
    allowed: list[str],
    primary: str,
    threshold: float,
    beam_size: int,
) -> list[Line]:
    lines: list[Line] = []

    for index, (start, end) in enumerate(windows, start=1):
        chunk = audio[start:end]
        offset = start / SAMPLE_RATE

        language, probability = pick_language(model, chunk, allowed, primary, threshold)
        segments, _ = model.transcribe(
            chunk,
            language=language,
            beam_size=beam_size,
            # The window is already known to be speech, and each is decoded in
            # isolation, so carrying text across windows would let one bad
            # decode seed the next.
            condition_on_previous_text=False,
        )

        for segment in segments:
            text = segment.text.strip()
            if not text:
                continue
            lines.append(
                Line(
                    start=offset + segment.start,
                    end=offset + segment.end,
                    language=language,
                    language_probability=round(probability, 3),
                    text=text,
                )
            )

        print(
            f"  [{index}/{len(windows)}] {offset:7.1f}s  {language}  p={probability:.2f}",
            file=sys.stderr,
        )

    return lines


def attach_speakers(
    lines: list[Line], audio_path: Path, hf_token: str | None
) -> str | None:
    """Label each line with a speaker. Returns why it could not, or None.

    Kept optional on purpose: pyannote's weights are gated on HuggingFace and
    need a one-time authenticated download, which some environments will not
    permit. Missing weights, a missing token, ungranted access and an empty
    offline cache all surface here, and by this point the transcription is
    already done -- so every one of them is reported and swallowed rather than
    allowed to discard the expensive part of the run.
    """
    try:
        from pyannote.audio import Pipeline
    except ImportError:
        return "pyannote.audio is not installed"

    try:
        pipeline = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-3.1", use_auth_token=hf_token
        )
        # pyannote answers an auth failure with None instead of raising.
        if pipeline is None:
            return "pyannote returned no pipeline (check the token and model access)"
        turns = [
            (turn.start, turn.end, speaker)
            for turn, _, speaker in pipeline(str(audio_path)).itertracks(yield_label=True)
        ]
    except Exception as error:
        return f"diarization failed ({type(error).__name__}: {error})"

    for line in lines:
        # Attribute the line to whoever actually holds most of it. Sampling a
        # single instant instead would hand the whole line to a two-word
        # interjection that happens to land there, and pyannote turns can
        # overlap, so the first match is whichever came out of the iterator.
        held: dict[str, float] = {}
        for start, end, speaker in turns:
            shared = min(line.end, end) - max(line.start, start)
            if shared > 0:
                held[speaker] = held.get(speaker, 0.0) + shared
        line.speaker = max(held, key=held.get) if held else None

    return None


def format_timestamp(seconds: float) -> str:
    ms = int(round(seconds * 1000))
    hours, ms = divmod(ms, 3_600_000)
    minutes, ms = divmod(ms, 60_000)
    secs, ms = divmod(ms, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{ms:03d}"


def beside(stem: Path, extension: str) -> Path:
    """Append an extension to the stem instead of replacing what looks like one.

    Path.with_suffix() would replace the last dotted part, so meeting.en.wav and
    meeting.ru.wav both reduce to meeting.srt and the second run silently
    overwrites the first -- and notes.v2.final.m4a loses ".final" outright.
    """
    return stem.parent / (stem.name + extension)


def write_outputs(lines: list[Line], stem: Path) -> list[Path]:
    srt = beside(stem, ".srt")
    with srt.open("w", encoding="utf-8") as handle:
        for index, line in enumerate(lines, start=1):
            speaker = f"[{line.speaker}] " if line.speaker else ""
            handle.write(
                f"{index}\n"
                f"{format_timestamp(line.start)} --> {format_timestamp(line.end)}\n"
                f"{speaker}{line.text}\n\n"
            )

    txt = beside(stem, ".txt")
    with txt.open("w", encoding="utf-8") as handle:
        for line in lines:
            speaker = f"[{line.speaker}] " if line.speaker else ""
            handle.write(f"{speaker}{line.text}\n")

    js = beside(stem, ".json")
    js.write_text(
        json.dumps([asdict(line) for line in lines], ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    return [srt, txt, js]


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    parser.add_argument("audio", type=Path, help="input recording, any format PyAV reads")
    parser.add_argument("--model", default="large-v3", help="size or path to a local model")
    parser.add_argument("--device", default="auto", choices=["auto", "cuda", "cpu"])
    parser.add_argument(
        "--compute-type",
        default="default",
        help="float16 on a GPU, int8 on CPU; 'default' lets CTranslate2 choose",
    )
    parser.add_argument(
        "--languages",
        default="ru,en,hy",
        help="languages to consider, comma separated; drop hy if no Armenian is spoken",
    )
    parser.add_argument(
        "--primary",
        default="ru",
        help="fallback for windows the classifier is unsure about",
    )
    parser.add_argument("--threshold", type=float, default=0.6, help="minimum confidence")
    parser.add_argument("--window", type=float, default=30.0, help="max window, seconds")
    parser.add_argument("--min-silence-ms", type=int, default=500)
    parser.add_argument("--speech-pad-ms", type=int, default=200)
    parser.add_argument(
        "--split-silence",
        type=float,
        default=0.7,
        help="a pause this long ends the window; a speaker change lives here",
    )
    parser.add_argument("--beam-size", type=int, default=5)
    parser.add_argument("--diarize", action="store_true", help="label speakers via pyannote")
    parser.add_argument("--hf-token", default=None, help="only used by --diarize")
    parser.add_argument("--out", type=Path, default=None, help="output stem")
    args = parser.parse_args()

    if not args.audio.exists():
        parser.error(f"no such file: {args.audio}")

    allowed = [lang.strip() for lang in args.languages.split(",") if lang.strip()]
    if args.primary not in allowed:
        parser.error(f"--primary {args.primary} is not in --languages {allowed}")

    # Make the output directory now, not after transcribing. A missing one is
    # otherwise discovered only at the write, which on a long recording is
    # hours of GPU time thrown away.
    stem = args.out or args.audio.with_suffix("")
    stem.parent.mkdir(parents=True, exist_ok=True)

    print(f"Decoding {args.audio}", file=sys.stderr)
    audio = decode_audio(str(args.audio), sampling_rate=SAMPLE_RATE)
    print(f"  {len(audio) / SAMPLE_RATE / 60:.1f} minutes", file=sys.stderr)

    windows = group_speech_windows(
        audio, args.window, args.min_silence_ms, args.speech_pad_ms, args.split_silence
    )
    if not windows:
        print("No speech found. Check that the file has an audio track.", file=sys.stderr)
        return 1
    speech = sum(end - start for start, end in windows) / SAMPLE_RATE
    print(f"  {len(windows)} windows, {speech / 60:.1f} minutes of speech", file=sys.stderr)

    print(f"Loading {args.model} on {args.device}", file=sys.stderr)
    model = WhisperModel(args.model, device=args.device, compute_type=args.compute_type)

    lines = transcribe(
        model, audio, windows, allowed, args.primary, args.threshold, args.beam_size
    )

    if args.diarize:
        problem = attach_speakers(lines, args.audio, args.hf_token)
        if problem:
            print(f"No speaker labels: {problem}", file=sys.stderr)

    written = write_outputs(lines, stem)

    counts: dict[str, int] = {}
    for line in lines:
        counts[line.language] = counts.get(line.language, 0) + 1
    spread = ", ".join(f"{lang} {n}" for lang, n in sorted(counts.items()))
    print(f"\n{len(lines)} lines ({spread})", file=sys.stderr)
    for path in written:
        print(f"  {path}", file=sys.stderr)

    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile app.py
#!/usr/bin/env python3
"""Drag-and-drop interface for the transcriber.

Same pipeline as `transcribe.py`, driven from a browser instead of a shell: drop
an audio or video file, get the transcript and the subtitle files back. Serves
both deployments -- `docker compose up ui` on a GPU box reaches it at
localhost:7860, and the Colab notebook launches the identical app on a borrowed
T4.

The model is loaded once and kept, so the first transcription pays the load cost
and later ones do not.
"""

from __future__ import annotations

import gc
import os
import sys
import tempfile
import time
from pathlib import Path

import gradio as gr
from faster_whisper import WhisperModel
from faster_whisper.audio import decode_audio

import transcribe as T

# Video containers are listed too: PyAV pulls the audio stream out of an mp4 or
# a mov, so a screen recording of a call works without converting it first.
ACCEPTED = [".mp3", ".m4a", ".wav", ".flac", ".ogg", ".opus", ".aac",
            ".mp4", ".mov", ".mkv", ".webm", ".avi"]

# The offline image bakes in exactly one model and sets HF_HUB_OFFLINE=1, so
# offering the others there would hand the user a dropdown whose other entries
# fail at load. Compose passes the baked model in; unset (Colab, a plain venv)
# means downloads work and the full list is honest.
MODEL_CHOICES = [
    name.strip()
    for name in os.environ.get(
        "STT_MODELS", "large-v3,large-v3-turbo,medium,small"
    ).split(",")
    if name.strip()
]

_models: dict[tuple[str, str, str], WhisperModel] = {}


def get_model(name: str, device: str, compute_type: str) -> WhisperModel:
    key = (name, device, compute_type)
    if key not in _models:
        _models[key] = WhisperModel(name, device=device, compute_type=compute_type)
    return _models[key]


def release_models() -> int:
    """Drop the cached models so their GPU memory goes back.

    Transcribing again reloads from disk -- seconds, not the original download.
    Worth doing between long files on a runtime that is also being used for
    something else; not worth doing between two clips.
    """
    freed = len(_models)
    _models.clear()
    gc.collect()
    return freed


def run(
    audio_file,
    languages: list[str],
    primary: str,
    model_name: str,
    threshold: float,
    window: float,
    split_silence: float,
    beam_size: int,
    progress=gr.Progress(),
):
    if audio_file is None:
        raise gr.Error("Drop a file first.")
    if not languages:
        raise gr.Error("Pick at least one language.")
    if primary not in languages:
        raise gr.Error(f"'{primary}' is the fallback language, so it has to be among the selected ones.")

    source = Path(audio_file)
    started = time.time()

    progress(0.02, desc="Reading the file")
    audio = decode_audio(str(source), sampling_rate=T.SAMPLE_RATE)
    minutes = len(audio) / T.SAMPLE_RATE / 60

    progress(0.06, desc="Finding speech")
    windows = T.group_speech_windows(audio, window, 500, 200, split_silence)
    if not windows:
        raise gr.Error("No speech found. Does the file actually have an audio track?")

    # Chosen after the file is read so the wait is attributable: a slow first
    # run is the download, not the audio.
    progress(0.10, desc=f"Loading {model_name}")
    device = "cuda" if _cuda_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    model = get_model(model_name, device, compute_type)

    lines: list[T.Line] = []
    unsure = 0
    for index, (start, end) in enumerate(windows):
        progress(
            0.10 + 0.88 * index / len(windows),
            desc=f"Transcribing {index + 1}/{len(windows)}",
        )
        produced = T.transcribe(
            model, audio, [(start, end)], languages, primary, threshold, beam_size
        )
        # Counted per window, not per line: one window emits several lines that
        # all carry its single classification, so counting lines would report
        # more uncertain windows than there are windows.
        if produced and produced[0].language_probability < 0.75:
            unsure += 1
        lines.extend(produced)

    if not lines:
        raise gr.Error("Speech was found but nothing was transcribed. Try a different model size.")

    progress(0.99, desc="Writing files")
    out_dir = Path(tempfile.mkdtemp(prefix="stt_"))
    written = T.write_outputs(lines, out_dir / source.stem)

    spread: dict[str, int] = {}
    for line in lines:
        spread[line.language] = spread.get(line.language, 0) + 1

    elapsed = (time.time() - started) / 60
    summary = "\n".join([
        f"{minutes:.1f} min of audio, {len(windows)} windows, {len(lines)} lines",
        f"languages: " + ", ".join(f"{k} {v}" for k, v in sorted(spread.items())),
        f"low-confidence windows: {unsure} of {len(windows)}"
        + (" — check those lines in the JSON" if unsure else ""),
        f"took {elapsed:.1f} min on {device}",
    ])

    transcript = "\n".join(
        f"[{line.language}] {line.text}" if len(spread) > 1 else line.text
        for line in lines
    )
    return transcript, [str(p) for p in written], summary


def _cuda_available() -> bool:
    try:
        import ctranslate2

        return ctranslate2.get_cuda_device_count() > 0
    except Exception:
        return False


def build() -> gr.Blocks:
    with gr.Blocks(title="STT_MultiLingva") as ui:
        gr.Markdown(
            "# STT_MultiLingva\n"
            "Drop a recording, get the transcript. Audio is processed by this "
            "process and is not sent anywhere.\n\n"
            "Whisper otherwise decides the language once and applies it to the "
            "whole file; this classifies every speech window separately, so a "
            "meeting that switches between languages does not come out in one."
        )

        with gr.Row():
            with gr.Column(scale=1):
                audio_file = gr.File(
                    label="Recording", file_types=ACCEPTED, type="filepath"
                )
                languages = gr.CheckboxGroup(
                    ["ru", "en", "hy"],
                    value=["ru", "en", "hy"],
                    label="Languages to expect",
                    info="Uncheck what is not spoken. Every extra language is another one a window can be misread as.",
                )
                primary = gr.Radio(
                    ["ru", "en", "hy"],
                    value="ru",
                    label="Fallback language",
                    info="Used for windows too short or unclear to classify.",
                )
                go = gr.Button("Transcribe", variant="primary")

                with gr.Accordion("Tuning", open=False):
                    model_name = gr.Dropdown(
                        MODEL_CHOICES,
                        value=MODEL_CHOICES[0],
                        label="Model",
                        info="large-v3 for quality. turbo is much faster but weaker on Armenian.",
                    )
                    threshold = gr.Slider(
                        0.3, 0.95, value=0.6, step=0.05,
                        label="Confidence threshold",
                        info="Below this a window falls back. Raise if wrong languages appear.",
                    )
                    split_silence = gr.Slider(
                        0.2, 2.0, value=0.7, step=0.1,
                        label="Pause that ends a window (s)",
                        info="Lower it if two speakers in different languages end up merged.",
                    )
                    window = gr.Slider(
                        5.0, 30.0, value=30.0, step=1.0,
                        label="Max window (s)",
                    )
                    beam_size = gr.Slider(
                        1, 10, value=5, step=1,
                        label="Beam size",
                        info="1 is a fast rough pass over a long file.",
                    )

            with gr.Column(scale=2):
                summary = gr.Textbox(label="Run", lines=4, interactive=False)
                transcript = gr.Textbox(label="Transcript", lines=26)
                downloads = gr.File(label="SRT / TXT / JSON")

        go.click(
            run,
            inputs=[audio_file, languages, primary, model_name, threshold,
                    window, split_silence, beam_size],
            outputs=[transcript, downloads, summary],
        )

    return ui


if __name__ == "__main__":
    share = "--share" in sys.argv
    build().queue().launch(
        server_name="0.0.0.0",
        server_port=7860,
        # Colab needs a tunnel to be reachable; a local box does not, and
        # opening one there would publish the app to the internet.
        share=share,
        max_file_size="2gb",
    )


In [ ]:
import sys
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import transcribe, app
print("tool ready")


## 4. Download the model once

Roughly 3 GB. Doing it here rather than on the first click means the interface
responds immediately instead of looking hung while it fetches.

In [ ]:
import app

MODEL = "large-v3"   # "large-v3-turbo" is much faster and weaker on Armenian

# Ask the app how it will choose, rather than assuming cuda/float16: the cache
# is keyed on that triple, so guessing wrong here warms an entry the interface
# never looks up and the first click downloads the model anyway.
DEVICE = "cuda" if app._cuda_available() else "cpu"
COMPUTE = "float16" if DEVICE == "cuda" else "int8"
if DEVICE == "cpu":
    print("No GPU visible. Runtime -> Change runtime type -> T4 GPU, then rerun from section 1.")

app.get_model(MODEL, DEVICE, COMPUTE)
print(f"{MODEL} loaded on {DEVICE} ({COMPUTE}) and cached")


## 5. The interface

Renders below this cell, and prints a `*.gradio.live` link that works from
another device — your phone, for instance — while this notebook keeps running.

Drop a file, pick the languages, press Transcribe. Audio and video both work:
PyAV pulls the audio stream out of an mp4, so a screen recording of a call needs
no conversion first.

**The share link is public to anyone holding it** for as long as the cell runs.
It is unlisted, not protected. Stop the cell when you are done.

In [ ]:
ui = app.build().queue()
ui.launch(share=True)


### Stopping it

Interrupting the cell (■) leaves the tunnel open until the runtime notices. This
closes it deliberately.

In [ ]:
ui.close()
print("interface stopped, share link dead")


## 6. Command line

For batch work, or when you want the exact flags in the output. Skip if section
5 did what you needed.

In [ ]:
from google.colab import files
import pathlib

uploaded = files.upload()
# files.upload() writes into the working directory, which section 3 moved into
# the clone -- so resolve against it rather than assuming a path.
AUDIO = str(pathlib.Path(next(iter(uploaded))).resolve())
UPLOADED = True   # this copy lives in the runtime and is ours to delete
print("using", AUDIO)


### Alternative: Drive

Instead of uploading. `UPLOADED = False` marks the file as yours rather than the
runtime's, so cleanup leaves it alone — deleting it would remove the original
recording from your Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# AUDIO = '/content/drive/MyDrive/meeting.m4a'
# UPLOADED = False


### Five minutes first

A short excerpt shows whether language detection is behaving before you commit
the whole file to a session Colab may reclaim mid-run.

In [ ]:
!ffmpeg -y -loglevel error -i "$AUDIO" -t 300 -ac 1 -ar 16000 /content/excerpt.wav
!python transcribe.py /content/excerpt.wav \
    --model large-v3 --device cuda --compute-type float16 \
    --languages ru,en,hy --primary ru --out /content/out/excerpt


In [ ]:
import json, collections
lines = json.load(open('/content/out/excerpt.json'))
print(collections.Counter(l['language'] for l in lines))
print()
for l in lines:
    if l['language_probability'] < 0.75:
        print(f"{l['start']:7.1f}s  {l['language']}  p={l['language_probability']:.2f}  {l['text'][:70]}")


### The whole file

On a T4, `large-v3` runs roughly 20–30x real time: a four-hour recording lands
in about ten minutes. Keep the tab open — Colab disconnects idle sessions.

In [ ]:
import time
start = time.time()
!python transcribe.py "$AUDIO" \
    --model large-v3 --device cuda --compute-type float16 \
    --languages ru,en,hy --primary ru --out /content/out/meeting
print(f"\n{(time.time() - start) / 60:.1f} minutes")


In [ ]:
from google.colab import files
for name in ['meeting.srt', 'meeting.txt', 'meeting.json']:
    files.download(f'/content/out/{name}')


### Clean up

Deleting the runtime's copy does not undo the upload — Google still received it.
This only limits how long it sits in the session. A file mounted from Drive is
never touched: it is the original, not a copy.

In [ ]:
import os

if UPLOADED:
    os.remove(AUDIO)
    print("removed the uploaded copy")
else:
    print(f"left {AUDIO} alone -- it is your Drive original, not a runtime copy")

if os.path.exists('/content/excerpt.wav'):
    os.remove('/content/excerpt.wav')
    print("removed the excerpt")


## Release the GPU

Colab bills a T4 for as long as one is assigned to you, not for how hard it is
working. An idle runtime with the notebook closed still spends quota — Colab
holds it for a while before reclaiming it on its own.

Two levels, depending on whether you are finished.

### Between files: free the memory, keep the session

Drops the cached model so its GPU memory goes back, without giving up the
runtime. Transcribing again reloads from local disk — seconds, not the original
three-gigabyte download.

In [ ]:
import app
print(f"released {app.release_models()} cached model(s)")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


### Finished: give the GPU back

Ends the runtime immediately and stops the quota. **Everything in the session is
destroyed** — the model, the uploads, and any transcript still sitting in
`/content/out`.

Before running this, confirm you have the files. Section 7's download cell puts
them in your browser's downloads; a file that only exists in `/content` is gone
the moment this executes.

In [ ]:
import os

pending = sorted(f for f in os.listdir("/content/out")
                 if os.path.isfile(f"/content/out/{f}")) if os.path.isdir("/content/out") else []

# Rerunning does not clear /content/out, so a plain "is it empty" check would
# refuse forever once anything had been written. The first run arms, the second
# releases -- which is what "rerun this cell" has to mean to be true.
armed = globals().get("_release_armed", False)

if pending and not armed:
    print("Still in the runtime and about to be destroyed:")
    for f in pending:
        print("   ", f)
    print("\nDownload them first if you have not. Rerun this cell to release anyway.")
    _release_armed = True
else:
    from google.colab import runtime
    print("Releasing the runtime. The GPU stops counting now.")
    runtime.unassign()


The cell refuses the first time if `/content/out` still holds files and
releases on a second run — rerunning is the confirmation, since downloading does
not empty the directory. That guard is deliberate: the GPU minutes are
replaceable, four hours of decoding is not.

To skip it outright: `from google.colab import runtime; runtime.unassign()`


## Armenian

`hy` is in the default set now — checked in the interface, and `--languages
ru,en,hy` on the command line. `large-v3` handles it out of the box at roughly
15.75% WER across dialects.

The default is a trade worth knowing about: the classifier's ranking is filtered
to the languages you allow, so a third entry is a third thing a Russian or
English window can be misread as, and Armenian scores noisily for being
low-resource. If Armenian appears where none was spoken, raise the confidence
threshold or uncheck `hy` for that recording.

If Armenian is a large share of the meeting rather than an occasional aside,
re-run the windows the JSON marks `hy` through a fine-tuned model
(`Chillarmo/whisper-large-v3-turbo-armenian`). Note `large-v3-turbo` is
distilled and noticeably weaker on Armenian than `large-v3`, so the speed trade
is worse here than it looks for Russian and English.
